# Oppitunti 05 - Agenttinen RAG


## Asennus

Tässä muistikirjassa havainnollistetaan Agentic RAG (Retrieval-Augmented Generation) -mallia käyttäen Microsoft Agent Frameworkia.

**Esivaatimukset:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI Search -palvelusi päätepiste
- `AZURE_SEARCH_API_KEY` — Azure AI Search -palvelusi API-avain
- Azure OpenAI -käyttöönotto konfiguroituna ympäristömuuttujien kautta
- Azure CLI kirjautuneena sisään (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mikä on Agentic RAG?

Perinteinen RAG noudattaa kiinteää putkea: hae dokumentteja, sitten luo vastaus. **Agentic RAG** menee pidemmälle antamalla agentille autonomia päättää **milloin** ja **miten** tietoa haetaan.

Agentic RAG:n avulla agentti voi:
- **Päättää**, tarvitaanko hakua ennen kysymykseen vastaamista
- **Valita**, mitä tietolähdettä tai työkalua käytetään hakuun
- **Arvioida** haetut tulokset ja tehdä jatkohakuja, jos ensimmäinen yritys ei riitä
- **Yhdistää** tietoa useista hakuvaiheista yhdeksi johdonmukaiseksi vastaukseksi

Tämä tekee agentista joustavamman ja tarkemman verrattuna staattiseen hae-sitten-luo -putkeen.


## Hakutyökalun luominen

Agentic RAG:ssa ulkoiset tietolähteet kääritään **työkaluiksi**, joita agentti voi kutsua tarpeen mukaan. Tämä antaa agentille mahdollisuuden käsitellä hakua yhtenä toimintona muiden joukossa sen sijaan, että se olisi pakollinen vaihe.

Alla määrittelemme matkustustietokannan ja teemme siitä työkalun, jota agentti voi käyttää kohdetietojen hakemiseen.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Rakentamassa RAG-agenttia

Nyt luomme agentin, jolle on annettu ohjeiksi **aina hakea tietoa ennen vastaamista**. Agentti käyttää `search_travel_knowledge`-työkalua perustaakseen vastauksensa tietopohjaan sen sijaan, että se luottaisi omaan koulutusdataansa.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratiivinen hakeminen — Maker-Checker-malli

Agentic RAG:n keskeinen etu on **iteratiivinen hakeminen**. Agentti voi suorittaa useita hakukierroksia varmistaakseen, tarkentaakseen tai laajentaakseen alkuperäisiä löytöjään — samanlaista kuin "maker-checker" -työnkulku:

1. **Tekijävaihe**: Agentti hakee alkuperäiset tiedot ja laatii vastauksen.
2. **Tarkastajavaihe**: Agentti suorittaa lisähakuja tarkistaakseen yksityiskohdat tai täyttääkseen aukot.

Alla agentilta kysytään kysymys, joka vaatii useiden kohteiden vertailua, mikä saa sen hakemaan useaan otteeseen.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Yhteenveto

Tässä oppitunnissa opit rakentamaan **Agentic RAG** -järjestelmän Microsoft Agent Frameworkin avulla:

- **Agentic RAG** antaa agenteille mahdollisuuden itse päättää, milloin haetaan tietoa, jolloin haku on dynaamista eikä kiinteää.
- **Työkalut datalähteinä**: Ulkoiset tietokannat (kuten Azure AI Search) kääritään työkaluiksi, joita agentti voi kutsua.
- **Iteratiivinen haku**: maker-checker-malli mahdollistaa agentille useiden hakukierrosten tekemisen — etsimisen, tarkistamisen ja tarkentamisen — ennen lopullisen vastauksen tuottamista.

Tuotantokäytössä korvaisit muistissa olevan `TRAVEL_KNOWLEDGE_BASE` -tietokannan oikealla Azure AI Search -indeksillä käsitelläksesi suuressa mittakaavassa matkailudokumenttien hakua.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
